# 🤖 LifeMate Agent - Kaggle Multi-GPU (T4 x 2) Evaluation
### Zalo AI Challenge 2026 - Voice-based Personal Assistant Agent

Notebook này hướng dẫn thiết lập môi trường và chạy đánh giá **LifeMate Agent** trên **Kaggle** sử dụng **2 GPU Tesla T4 (T4 x 2)**.

#### ⚡ Điểm nổi bật:
- **Tận dụng tối đa T4 x 2 (Data-Parallelism)**: Chạy 2 tiến trình song song trên `cuda:0` và `cuda:1`, xử lý đồng thời các mẫu dự đoán giúp **tăng tốc độ gấp đôi (giảm 50% thời gian đánh giá)**.
- **Chuẩn thể lệ ZAC 2026**: Tính điểm chuẩn theo `state_match`, Kuhn Maximum Bipartite Matching, và Token-F1 (> 0.6) cho các trường fuzzy.
- **Hỗ trợ Model-Parallelism (`--gpus auto`)**: Tự động shard model lớn (ví dụ 7B/14B) qua cả 2 GPU khi cần.
- **Tự động nhận diện dữ liệu**: Tự tìm data trong `/kaggle/input` hoặc thư mục local `./data`.

## 1. Kiểm tra phần cứng GPU (Tesla T4 x 2)

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
device_count = torch.cuda.device_count()
print(f"Detected GPUs: {device_count}")
for i in range(device_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  - GPU {i}: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")
if device_count < 2:
    print("⚠️ Lưu ý: Bạn đang bật 1 GPU hoặc CPU. Trong Settings bên phải của Kaggle, hãy chọn Accelerator là 'GPU T4 x 2' để tận dụng tối đa!")

## 2. Cài đặt các thư viện cần thiết
Cài đặt các gói hỗ trợ mô hình LLM, 4-bit NF4 Quantization (BitsAndBytes), ASR và âm thanh.

In [ ]:
!pip install -q transformers==5.21.0 accelerate bitsandbytes torchvision soundfile librosa

## 3. Thiết lập Repository & Thư mục làm việc
Đoạn mã dưới đây đảm bảo thư mục làm việc và `PYTHONPATH` được cấu hình chính xác dù bạn tải repository từ GitHub hoặc upload notebook vào Kaggle.

In [ ]:
import os
import sys
from pathlib import Path

# Kiểm tra nếu repo đã có sẵn hoặc cần clone
if Path("evaluate.py").is_file():
    REPO_DIR = Path(".").resolve()
elif Path("/kaggle/working/LifeMate").is_dir():
    REPO_DIR = Path("/kaggle/working/LifeMate").resolve()
else:
    print("Cloning repository...")
    !git clone https://github.com/NT-Loi/LifeMate.git /kaggle/working/LifeMate
    REPO_DIR = Path("/kaggle/working/LifeMate").resolve()

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"Thư mục làm việc: {os.getcwd()}")

## 4. Kiểm tra dữ liệu ZAC 2026
`config.py` đã được tích hợp tự động dò tìm dữ liệu trong `/kaggle/input` nếu bạn add dataset vào Kaggle, hoặc dùng thư mục `./data` có sẵn trong repo.

In [ ]:
from config import MODE, DATA_ROOT
from huggingface_hub import snapshot_download
from models.llm.qwen import MODEL_REVISIONS

MODEL_ID = "Qwen/Qwen3.5-2B"  # Change to another supported model ID to compare models.
snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISIONS[MODEL_ID])


print(f"📁 Thư mục dữ liệu: {DATA_ROOT}")
for split in ["train", "public_test"]:
    cfg = MODE[split]
    cat_exists = Path(cfg["catalog_path"]).is_file()
    data_exists = Path(cfg["data_path"]).is_file()
    tools_exists = Path(cfg["tools_dir"]).is_dir()
    print(f"[{split.upper()}] Data: {'✅' if data_exists else '❌'} | Catalog: {'✅' if cat_exists else '❌'} | zaic_tools: {'✅' if tools_exists else '❌'}")

## 5. Chạy Đánh giá Song Song trên T4 x 2 (CLI)
Sử dụng cờ `--gpus 0,1` để kích hoạt 2 worker processes độc lập trên GPU 0 và GPU 1.
- Worker 0 chạy trên GPU 0 nhận các mẫu số chẵn (0, 2, 4, ...).
- Worker 1 chạy trên GPU 1 nhận các mẫu số lẻ (1, 3, 5, ...).
- Cả 2 GPU hoạt động 100% công suất song song, giảm 50% thời gian thực thi!

In [ ]:
# Đánh giá thử 10 mẫu đầu tiên trên cả 2 GPU T4 (0 và 1)
!python evaluate.py --mode train --num-samples 10 --gpus 0,1 --model-id $MODEL_ID

## 6. Chạy Đánh giá trực tiếp bằng Python API & Trực quan hóa kết quả
Bạn cũng có thể gọi hàm `run_evaluation` trực tiếp từ notebook để lấy kết quả dạng DataFrame và phân tích chi tiết.

In [ ]:
import json
import pandas as pd
from config import MODE
from evaluate import run_evaluation

# Đọc 10 mẫu từ train.jsonl
samples = []
with open(MODE["train"]["data_path"], "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))
        if len(samples) >= 10:
            break

# Chạy đánh giá trên 2 GPU
results = run_evaluation(samples=samples, mode="train", gpus=[0, 1], model_id=MODEL_ID)

# Hiển thị bảng tổng kết DataFrame
df = pd.DataFrame(results)
cols = ["id", "state_match", "p", "r", "f1", "score"]
display_cols = [c for c in cols if c in df.columns]
df_styled = df[display_cols].style.map(
    lambda v: "background-color: #d4edda; color: #155724; font-weight: bold;" if v is True else (
        "background-color: #f8d7da; color: #721c24; font-weight: bold;" if v is False else ""
    ),
    subset=["state_match"]
)
df_styled

## 7. Đánh giá chuyên sâu 1 mẫu cụ thể (Single Sample Diagnostic)
Khi cần debug chi tiết từng bước ReAct loop của một sample cụ thể (ví dụ: `train_0001`):

In [ ]:
!python evaluate.py --id train_0001 --model-id $MODEL_ID

## 8. Xem các tệp kết quả đã lưu (Submission & Metrics Artifacts)
Quá trình đánh giá tự động lưu lại 4 tệp trong thư mục kết quả (`eval_results/latest/`):
1. `predictions.jsonl`: File dự đoán theo chuẩn nộp bài ZAC 2026 (`id`, `tool_calls`, `state_hash`).
2. `eval_details.jsonl`: Chi tiết điểm và so sánh từng sample, bao gồm cả mảng `messages` chứa toàn bộ lịch sử hội thoại.
3. `conversations.txt`: Toàn bộ lịch sử hội thoại multi-turn dạng văn bản trực quan (System Prompt, User, Tool results, Assistant, Evaluation Summary).
4. `eval_summary.json`: Báo cáo tổng kết chỉ số (Accuracy, Precision, Recall, F1, Mean Score).

In [ ]:
# Liệt kê các tệp kết quả mới nhất
!ls -lh eval_results/latest/

# Đọc thử 2 dòng đầu file dự đoán predictions.jsonl
!head -n 2 eval_results/latest/predictions.jsonl

# Đọc thử lịch sử hội thoại của sample đầu tiên trong conversations.txt
!head -n 50 eval_results/latest/conversations.txt

# Hiển thị báo cáo eval_summary.json
import json
if Path("eval_results/latest/eval_summary.json").is_file():
    with open("eval_results/latest/eval_summary.json", "r", encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

## 9. Chạy Đánh giá toàn bộ tập dữ liệu (Full Train Evaluation)
Chạy đánh giá toàn bộ các mẫu của tập train (`--num-samples -1`) trên cả 2 GPU T4.
> 💡 Bỏ comment dòng bên dưới khi bạn sẵn sàng chạy full tập train.

In [ ]:
# !python evaluate.py --mode train --num-samples -1 --gpus 0,1 --model-id $MODEL_ID

## 10. Chọn mô hình
Đổi `MODEL_ID` ở ô tải model để so sánh các ID được đăng ký trong `models/llm`. Các ô đánh giá sẽ dùng cùng ID và revision đã tải vào cache.

In [ ]:
# Ví dụ: chạy đánh giá với model đã chọn ở ô trên.
# !python evaluate.py --num-samples 5 --gpus auto --model-id $MODEL_ID